# 01f -- dim_season Seed

**Purpose**: Calendar spine for the new event-sourced facts (ADR-0004). One row
per league season. New facts key `season -> season_id`; `dim_draft_pick.draft_season`
-> `season_id` too.

**Key**: `season_id` (string `"2026-2027"`).

**Horizon**: current + 2 (2026-2027, 2027-2028, 2028-2029) -- matches the
`dim_draft_pick` current+2 pick horizon (PLAN.md / ADR-0004 grill 2026-06-14).

**Dates**:
- `season_fantasy_start_date` = Mar 1 of the start year (league year opens).
- `season_fantasy_end_date`   = last day of Feb of the end year.
- `season_nfl_start_date` / `season_nfl_end_date` = public NFL schedule,
  **nullable** until known (left null at seed; backfilled when the schedule drops).

**Output**: `data/dim_season.parquet`

In [ ]:
# ---- Setup & Config (shared module) ----------------------------------------
import sys
from pathlib import Path
for _p in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import CFG, DATA

import pandas as pd
from datetime import date, timedelta

## 1 -- Build

In [ ]:
# ---- Build dim_season ------------------------------------------------------
# Horizon = current + 2. CFG.draft_year is the current league start year (2026).
HORIZON = 3
START_YEAR = CFG.draft_year  # 2026

# Seed-time themes (descriptive only; extend as seasons gain identity).
_THEMES = {2026: "Startup Draft (inaugural)"}


def _last_day_of_feb(end_year: int) -> date:
    # Last day of February = (Mar 1) - 1 day; handles leap years automatically.
    return date(end_year, 3, 1) - timedelta(days=1)


def build_dim_season(start_year: int, horizon: int) -> pd.DataFrame:
    rows = []
    for sy in range(start_year, start_year + horizon):
        ey = sy + 1
        rows.append({
            "season_id": f"{sy}-{ey}",
            "season_start_year": sy,
            "season_end_year": ey,
            "season_fantasy_start_date": date(sy, 3, 1),
            "season_fantasy_end_date": _last_day_of_feb(ey),
            "season_nfl_start_date": pd.NaT,   # nullable until schedule known
            "season_nfl_end_date": pd.NaT,     # nullable until schedule known
            "theme": _THEMES.get(sy, pd.NA),
        })
    df = pd.DataFrame(rows)
    # datetime64 for the date columns (NaT-friendly, Power BI date type).
    for c in ("season_fantasy_start_date", "season_fantasy_end_date",
              "season_nfl_start_date", "season_nfl_end_date"):
        df[c] = pd.to_datetime(df[c])
    return df


dim_season = build_dim_season(START_YEAR, HORIZON)
out_path = DATA / "dim_season.parquet"
dim_season.to_parquet(out_path, index=False)
print(f"dim_season: {len(dim_season)} rows -> {out_path}")
dim_season

## 2 -- Validation

In [ ]:
# ---- Validation ------------------------------------------------------------
df = pd.read_parquet(DATA / "dim_season.parquet")
assert df["season_id"].is_unique, "season_id must be unique (PK)"
assert (df["season_end_year"] == df["season_start_year"] + 1).all()
# Fantasy window: Mar 1 -> last day of Feb next year.
assert (df["season_fantasy_start_date"].dt.month == 3).all()
assert (df["season_fantasy_start_date"].dt.day == 1).all()
assert (df["season_fantasy_end_date"].dt.month == 2).all()
print("dim_season OK")
print(df.to_string(index=False))